# Capstone 2: Restaurant Chain ETL

This is a proper data engineering problem. Three franchise locations,
three different data formats, one stakeholder who wants a weekly
performance report by Friday afternoon.

You will extract from CSV, JSON, and SQLite. You will validate with
Pydantic. You will design a star schema. You will load into DuckDB
and produce the report. The brief is short. The decisions are yours.

## The brief

The Hearthstone Restaurant Group operates three franchise locations:

| Branch | Location | Data format | Quirks |
|--------|----------|-------------|--------|
| North  | Sheffield | CSV | UK date format (dd/mm/yyyy), GBP, flat structure |
| South  | Brighton | JSON | ISO timestamps, nested items array, GBP |
| West   | Abergavenny | SQLite | Mixed date formats, some EUR prices, duplicate orders |

Franchise HQ needs a consolidated weekly performance report. They
want answers to four questions:

1. Which branch has the highest revenue?
2. What are the top-selling items across all branches?
3. How do weekday vs weekend sales compare?
4. Which branch has the most data quality issues?

Your job is to build the pipeline that makes that report possible.

## The data

All files are in the `../data/` directory:

- `restaurant_north.csv` -- ~800 order rows, one row per item ordered.
  Columns: order_id, order_date, item_name, quantity, unit_price,
  total, payment_method, server_name.

- `restaurant_south.json` -- ~600 orders as a JSON array. Each order
  has a nested `items` list with name, qty, and price per item.

- `restaurant_west.sqlite` -- A SQLite database with two tables:
  `orders` (order_id, date, total, currency, payment_method) and
  `order_items` (order_id, item_name, quantity, price). Contains
  duplicate orders and a mix of GBP and EUR prices.

Each source has its own idiosyncrasies. That is realistic. In the
real world, you never get three sources that agree on anything.

## Setup

In [ ]:
%pip install -q duckdb pydantic pyarrow matplotlib

In [ ]:
import pandas as pd
import json
import sqlite3
import duckdb
import matplotlib.pyplot as plt
from datetime import datetime

---

## Step 1: Extract

Pull data from all three sources into pandas DataFrames. The goal
is to get each source into a consistent tabular shape with at
minimum these columns:

- `order_id`
- `order_date` (as a date, not a string)
- `item_name`
- `quantity`
- `unit_price`
- `branch` (you will need to add this -- "North", "South", or "West")

The tricky one is the South branch, because items are nested inside
each order. You will need to flatten that structure.

The West branch requires reading from SQLite and joining the orders
and order_items tables.

### Task: Extract from North (CSV)

In [ ]:
# Your code here


### Task: Extract from South (JSON)

Remember: each order has a nested `items` array. You need to
flatten this so each row is one item within one order.

In [ ]:
# Your code here


### Task: Extract from West (SQLite)

Connect to the SQLite database, join the two tables, and pull
into a DataFrame. Watch out for the `currency` column -- you will
deal with EUR prices in the transform step.

In [ ]:
# Your code here


---

## Step 2: Validate with Pydantic

Before you transform the data, define what a valid order line
should look like. This is where Pydantic earns its keep.

Create a Pydantic model (or models) that captures your expectations:

- `order_id` should be a non-empty string
- `order_date` should be a valid date
- `item_name` should be a non-empty string
- `quantity` should be a positive integer
- `unit_price` should be a positive number
- `branch` should be one of "North", "South", "West"

Run each source's DataFrame through validation. Capture and count
the errors. This will answer question 4 from the brief (which
branch has the most data quality issues).

### Task: Define the Pydantic model

In [ ]:
from pydantic import BaseModel, field_validator


class OrderLine(BaseModel):
    """A single line item from any branch."""
    # Your code here -- define the fields and any validators
    pass

### Task: Validate each source

Write a function that takes a DataFrame and attempts to validate
each row against your Pydantic model. Return two things: a list
of valid records and a list of errors (with enough detail to
diagnose each failure).

In [ ]:
def validate_dataframe(df, model_class):
    """Validate each row of a DataFrame against a Pydantic model.

    Returns:
        valid_records: list of validated dicts
        errors: list of dicts with row index and error detail
    """
    # Your code here
    pass

### Task: Run validation and report results

Run validation on each branch's data. Print a summary:
how many valid rows, how many errors, what kinds of errors.

In [ ]:
# Your code here


---

## Step 3: Transform

Now clean and standardise the validated data. Things to address:

- **Dates:** All branches should have dates in the same format.
  Parse the North branch's `dd/mm/yyyy`, the South's ISO timestamps,
  and the West's mixed formats into proper datetime objects.

- **Currency:** The West branch has some prices in EUR. Convert
  these to GBP. Use a fixed exchange rate of 0.86 (1 EUR = 0.86 GBP).
  In production you would call an API. Here a constant is fine.

- **Deduplication:** The West branch has duplicate orders. Remove them.

- **Combine:** Merge all three branches into a single DataFrame.

After this step you should have one clean, unified dataset with
consistent types and no duplicates.

### Task: Parse and standardise dates

In [ ]:
# Your code here


### Task: Convert EUR to GBP for the West branch

In [ ]:
EUR_TO_GBP = 0.86

# Your code here


### Task: Deduplicate the West branch

The West branch has duplicate rows in the orders table. Identify
and remove them. Think carefully about which columns to use for
deduplication.

In [ ]:
# Your code here


### Task: Combine all branches

Concatenate the three cleaned DataFrames into one. Verify the
total row count makes sense.

In [ ]:
# Your code here


---

## Step 4: Model -- Star Schema Design

Before loading into the analytics layer, design a star schema.
The fact table captures what happened (orders). The dimension
tables capture the context (when, where, what).

Here is a suggested design. You may adjust it.

**fact_orders:**
- order_line_id (surrogate key)
- order_id
- date_key (FK to dim_date)
- branch_key (FK to dim_branch)
- menu_item_key (FK to dim_menu_item)
- quantity
- unit_price_gbp
- line_total_gbp

**dim_date:**
- date_key
- full_date
- day_of_week
- is_weekend
- week_number
- month
- month_name

**dim_branch:**
- branch_key
- branch_name
- location
- cuisine_style

**dim_menu_item:**
- menu_item_key
- item_name
- item_name_clean (standardised)

The dimension tables are small. Build them from the data plus any
enrichment you want to add. The fact table is built by joining
your combined DataFrame against the dimension keys.

### Task: Build dimension tables

Create DataFrames for `dim_date`, `dim_branch`, and `dim_menu_item`.
Generate surrogate keys (simple integers are fine).

In [ ]:
# Your code here


### Task: Build the fact table

Join your combined order data against the dimension tables to
replace natural keys with surrogate keys. Calculate
`line_total_gbp` as `quantity * unit_price_gbp`.

In [ ]:
# Your code here


---

## Step 5: Load into DuckDB

Save your star schema tables as Parquet files, then query them
with DuckDB. This gives you the columnar analytics performance
that stakeholders expect.

The workflow:
1. Save each table as a `.parquet` file
2. Create a DuckDB connection
3. Query the Parquet files directly -- DuckDB handles this natively

### Task: Save to Parquet and connect DuckDB

In [ ]:
# Your code here


---

## Step 6: The Report

Answer the four questions from the brief. For each one, write
a DuckDB query and present the results clearly. A chart or two
would not go amiss -- this is for stakeholders, not engineers.

### Report Question 1: Which branch has the highest revenue?

Join fact_orders to dim_branch. Sum `line_total_gbp` by branch.

In [ ]:
# Your code here


### Report Question 2: What are the top-selling items across all branches?

Join fact_orders to dim_menu_item. Sum quantity by item. Show top 10.

In [ ]:
# Your code here


### Report Question 3: How do weekday vs weekend sales compare?

Join fact_orders to dim_date. Group by `is_weekend`. Compare total
revenue and average order value.

**Bonus:** Break this down by branch as well. Do all branches show
the same weekday/weekend pattern?

In [ ]:
# Your code here


### Report Question 4: Which branch has the most data quality issues?

You collected validation errors in Step 2. Present that data here.
Which branch had the most errors? What types of errors? What would
you recommend to franchise HQ about data submission standards?

In [ ]:
# Your code here -- present the validation error summary


---

## Reflection

You have built a multi-source ETL pipeline with validation, a star
schema, and an analytics layer. That is not a tutorial exercise.
That is the kind of work data engineers do every week.

### Your turn

**Think about these questions and write your answers below.**

**1. What would make this pipeline idempotent?**

If you ran it twice on the same data, would you get duplicate
records? What would you change?

*Your answer:*

...

**2. What happens when the West branch fixes their EUR problem?**

Schema evolution in action. Your pipeline hardcodes a currency
conversion. What if the source changes? How do you design for
that?

*Your answer:*

...

**3. What would you monitor in production?**

Row counts, null rates, validation failure rates, schema drift,
latency -- what matters most for this particular pipeline?

*Your answer:*

...

**4. How would you handle a fourth branch?**

The restaurant group is expanding. A new branch in Edinburgh sends
data as XML. How much of your pipeline would you need to change?

*Your answer:*

...

---

## Summary

This capstone tested the full ETL skill set: extraction from
multiple formats, validation with Pydantic, transformation
including currency conversion and deduplication, dimensional
modelling, and analytics with DuckDB.

The messy reality of multi-source data is something you will
encounter constantly in professional data engineering. The sources
never agree. The formats are never consistent. The stakeholder
always wants the report by Friday. The skill is not in knowing
any single tool perfectly -- it is in stitching them together into
a pipeline that works reliably and produces trustworthy results.

In the final capstone, we move into newer territory: text
processing, vector search, and retrieval-augmented generation.
The guidance drops to a minimum. You will get a brief and data.
Everything else is up to you.